# Pipeline A v2 — 1-to-5 rating instead of yes/no

**Why:** v1 (yes/no) said "yes" to 96% of Missouri and 99% of Haryana collages, and its reasons copied the prompt's
gully description word for word. Here the model gives a **score from 1 to 5**, with no detailed gully description to copy.
We then pick the cut-off on Missouri, and report **AUC**, a score that a yes-bias can't inflate.

**Two modes** (cell 5, `MODE`):
- `"test"`: try 2 prompt versions on 150 Missouri collages (75 gully + 75 no gully), ~35 min. **Run this first.**
- `"full"`: run the winning version on all Missouri + Haryana, ~1.5 h.

**Settings (right panel):** Accelerator → GPU T4 x2 · Internet → On · Input → `gully-collages`.
Download the CSVs from **Output → /kaggle/working** before stopping the session.

In [ ]:
# 1. GPU check
!nvidia-smi

In [ ]:
# 2. Install Ollama
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. Start Ollama server
import subprocess, time, requests
subprocess.Popen("ollama serve > /kaggle/working/ollama.log 2>&1", shell=True)
for _ in range(60):
    try:
        requests.get("http://localhost:11434", timeout=2); print("Ollama server is running"); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Ollama did not start - see /kaggle/working/ollama.log")

In [ ]:
# 4. Download the model
!ollama pull qwen2.5vl:7b

In [ ]:
# 5. Settings + data
import base64, csv, glob, json, os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

MODE = "test"              # "test" first, then change to "full"
FULL_VARIANT = "evidence"  # which prompt to use in full mode (pick the winner from the test)
TEST_PER_CLASS = 75        # test mode: 75 gully + 75 no-gully Missouri collages

MODEL = "qwen2.5vl:7b"
OUT_DIR = "/kaggle/working"

def find_dataset(name):
    hits = [p for p in glob.glob("/kaggle/input/**/collages.csv", recursive=True) if name in p.lower()]
    if not hits:
        raise FileNotFoundError(f"No collages.csv for '{name}' - did you add the gully-collages dataset?")
    folder = os.path.dirname(sorted(hits, key=len)[0])
    df = pd.read_csv(folder + "/collages.csv", dtype={"label": str})
    df["path"] = folder + "/" + df["collage"]
    df = df[df["path"].map(os.path.exists)].reset_index(drop=True)
    print(f"{name}: {len(df)} collages")
    return df

missouri = find_dataset("missouri")
haryana = find_dataset("haryana")

if MODE == "test":
    missouri = (missouri[missouri["label"].isin(["0", "1"])]
                .groupby("label").sample(n=TEST_PER_CLASS, random_state=42)
                .reset_index(drop=True))
    print("Test subset:", missouri["label"].value_counts().to_dict())

In [ ]:
# 6. Two prompt versions. Neither describes what a gully looks like in detail (v1's mistake).
INTRO = ('The image is a collage of {n} overhead images of the SAME agricultural field, taken on different dates. '
         'Each tile has its date or image number in the top-left corner. Ignore any grey "no image" tile or text tile.\n\n'
         'An ephemeral gully is a small erosion channel cut by rainwater runoff, which farmers fill in during tillage. '
         'Many fields have no gully, so only give a high score if you see clear evidence.\n\n')

SCALE = ('Score from 1 to 5 how likely it is that this field contains an ephemeral gully:\n'
         '1 = almost certainly no, 2 = probably no, 3 = unsure, 4 = probably yes, 5 = almost certainly yes.')

VARIANTS = {
    # A: just the score, no explanation to talk itself into a "yes"
    "short": {
        "prompt": INTRO + SCALE,
        "schema": {"type": "object",
                   "properties": {"score": {"type": "integer", "enum": [1, 2, 3, 4, 5]}},
                   "required": ["score"]},
    },
    # B: first list the line features and say what each one is, then score
    "evidence": {
        "prompt": INTRO + ('First, list the linear features you can see (if any) and say what each most likely is, '
                           'e.g. crop rows, field boundary, road, tractor track, canal, or erosion channel. '
                           'Then give your score.\n\n') + SCALE,
        "schema": {"type": "object",
                   "properties": {"features": {"type": "string"},
                                  "score": {"type": "integer", "enum": [1, 2, 3, 4, 5]}},
                   "required": ["features", "score"]},
    },
}

def ask(variant, image_path, n_images):
    v = VARIANTS[variant]
    with open(image_path, "rb") as f:
        img = base64.b64encode(f.read()).decode()
    r = requests.post("http://localhost:11434/api/chat", timeout=600, json={
        "model": MODEL,
        "messages": [{"role": "user", "content": v["prompt"].format(n=n_images), "images": [img]}],
        "format": v["schema"], "stream": False,
        "options": {"temperature": 0, "num_ctx": 4096, "seed": 42},
    })
    r.raise_for_status()
    raw = r.json()["message"]["content"]
    try:
        out = json.loads(raw)
        score = int(out["score"])
        reason = str(out.get("features", ""))
    except Exception:
        score, reason = "", raw
    return score, reason, raw

# quick check
row = missouri.iloc[0]
for name in VARIANTS:
    t = time.time()
    s, reason, _ = ask(name, row["path"], row["n_images"])
    print(f"[{name}] tile {row['tile_id']} (truth={row['label']}) -> score={s} ({time.time()-t:.1f}s)  {reason[:200]}")

In [ ]:
# 7. Run (resumable - re-run this cell if it stops)
def run(variant, name, df):
    out_csv = f"{OUT_DIR}/pipeline_a2_{variant}_{name}.csv"
    done = set(pd.read_csv(out_csv)["tile_id"]) if os.path.exists(out_csv) else set()
    todo = df[~df["tile_id"].isin(done)]
    new_file = not os.path.exists(out_csv)
    with open(out_csv, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(["tile_id", "label", "score", "reason", "seconds", "raw"])
        for _, row in tqdm(todo.iterrows(), total=len(todo), desc=f"{variant}/{name}"):
            t = time.time()
            try:
                s, reason, raw = ask(variant, row["path"], row["n_images"])
            except Exception as e:
                s, reason, raw = "", f"ERROR: {e}", ""
            w.writerow([row["tile_id"], row["label"] if pd.notna(row["label"]) else "", s, reason,
                        round(time.time() - t, 1), raw])
            f.flush()
    return pd.read_csv(out_csv, dtype={"label": str})

results = {}
if MODE == "test":
    for v in VARIANTS:
        results[(v, "missouri")] = run(v, "missouri", missouri)
else:
    results[(FULL_VARIANT, "missouri")] = run(FULL_VARIANT, "missouri", missouri)
    results[(FULL_VARIANT, "haryana")] = run(FULL_VARIANT, "haryana", haryana)

In [ ]:
# 8. Results
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

def evaluate(df):
    d = df[df["label"].isin(["0", "1"]) & pd.to_numeric(df["score"], errors="coerce").notna()].copy()
    y, s = d["label"].astype(int).values, d["score"].astype(int).values
    print(f"  scored: {len(d)}/{len(df)}   score counts: {pd.Series(s).value_counts().sort_index().to_dict()}")
    print(f"  mean score - gully: {s[y==1].mean():.2f}   no gully: {s[y==0].mean():.2f}")
    auc = roc_auc_score(y, s) if len(set(y)) == 2 else float("nan")
    print(f"  AUC: {auc:.3f}   (0.5 = guessing, 1.0 = perfect)")
    best = None
    for t in [2, 3, 4, 5]:
        p = (s >= t).astype(int)
        row = (t, accuracy_score(y, p), balanced_accuracy_score(y, p), f1_score(y, p, zero_division=0))
        print(f"  'gully' if score >= {t}: acc={row[1]:.3f}  balanced acc={row[2]:.3f}  F1={row[3]:.3f}  yes-rate={p.mean():.2f}")
        if best is None or row[2] > best[2]:
            best = row
    print(f"  majority baseline acc: {max(y.mean(), 1-y.mean()):.3f}")
    return auc, best

summary = []
for (v, name), df in results.items():
    if name != "missouri":
        continue
    print(f"=== {v} / missouri ===")
    auc, best = evaluate(df)
    summary.append({"variant": v, "AUC": round(auc, 3), "best_threshold": best[0],
                    "acc": round(best[1], 3), "balanced_acc": round(best[2], 3), "F1": round(best[3], 3)})
    print()
print(pd.DataFrame(summary).to_string(index=False))
print("\nv1 (yes/no) on Missouri: acc 0.586, balanced acc 0.521, yes-rate 0.96")

if MODE == "full":
    thr = summary[0]["best_threshold"]
    h = results[(FULL_VARIANT, "haryana")]
    hs = pd.to_numeric(h["score"], errors="coerce")
    print(f"\n=== Haryana (threshold {thr} chosen on Missouri) ===")
    print("score counts:", hs.value_counts().sort_index().to_dict())
    print(f"predicted gully: {(hs >= thr).sum()} / {hs.notna().sum()}")